# Domovina.tv — 🎭 Sortformer DIARIZE-ONLY *(Colab T4-friendly)*

**Pass 2 od 2-pass scale workflow-a.** Pokreće SAMO Sortformer diarizaciju nad postojećim `.canary.srt` datotekama.

```
Pass 1  (G4 / Pro+):  colab_canary/domovina_tv_fetch.ipynb        → .canary.srt        (~13s/file)
Pass 2  (T4 / FREE):  OVAJ NOTEBOOK                                → .sortformer.diarized.srt  (~3-5s/file)
```

## Zašto T4 staje (a u kombiniranom notebook-u nije)?

| Komponenta | VRAM peak | Staje na T4 (16 GB)? |
|---|---|---|
| Canary 1B v2 (transkripcija) | ~26 GB | ❌ OOM |
| Streaming Sortformer 4spk v2.1 | **~2 GB** | ✅ Lako |

Canary je taj koji zahtijeva G4. Bez njega — sve teče na free Colab T4 instanci. Ovo te troši **0 compute units** (T4 je free tier).

## Kad koristiti Pass 2 vs combined notebook (`domovina_tv_sortformer.ipynb`)

| Slučaj | Notebook |
|---|---|
| Imaš `.canary.srt` na Drive-u (već si pokrenuo Pass 1 ili stable canary pipeline) | **Ovaj** (jeftinije, brže) |
| Nemaš `.canary.srt` (greenfield batch) | `domovina_tv_sortformer.ipynb` (sve u jednom prolasku na G4) |
| Skaliranje na tisuće fajlova: razdvojeni runtime-ovi | Pass 1 + Pass 2 (paralelno na 2 računa) |

## Output

`{wav}.sortformer.diarized.srt` — **bit-identičan** onome što generira combined notebook (isti Canary tekst iz `.canary.srt`, isti Sortformer speakeri, isti merge algoritam). Drop-in kompatibilan s ostalim alatima.

## ⚠️ LICENCA

Streaming Sortformer 4spk v2.1 je pod [NVIDIA Open Model License](https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-open-model-license/) (komercijalno OK uz uvjete). NE mijenjati na CC-BY-NC v1/v2 varijantu.

## Kako koristiti

1. **Runtime**: Runtime → Change runtime type → **T4** (free) — A100/L4/G4 rade ali su skuplji bez razloga.
2. **HF token**: Secrets panel → `HF_TOKEN` (potreban za skidanje Sortformer modela).
3. **Drive folder**: WAV+`.canary.srt` u `MyDrive/domovina_fetch_data/canary_wav/{kanal}/...`.
4. **Runtime → Run all**. Prvi put restart kernela nakon instalacije — pokreni opet.

---


## 0. Konfiguracija


In [ ]:
# ─── Drive locations (isto kao stable canary pipeline) ─────────────────────
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_DATA_DIR    = "MyDrive/domovina_fetch_data/canary_wav"
INPUT_DIR         = f"{DRIVE_MOUNT_POINT}/{DRIVE_DATA_DIR}"

# ─── Batch limits ────────────────────────────────────────────────────────────
LIMIT             = None   # int ili None — npr. 5 za testiranje
DRY_RUN           = False  # True: samo prikaz, bez obrade

# ─── Paralelizacija ─────────────────────────────────────────────────────────
# Koliko fajlova ide paralelno kroz Sortformer u JEDNOM GPU forward pass-u.
# T4 sweet spot: 4-8. Veći = brže, ali raste System RAM linearno
# (~330 MB × batch za 3h fajlove). T4 ima 12.7 GB sys RAM → batch=8 je sigurno.
# Ako CUDA OOM ili sys RAM blizu limita → smanji BATCH_SIZE.
BATCH_SIZE        = 4

# ─── Auto-shutdown ───────────────────────────────────────────────────────────
# True: nakon završetka batcha gasi Colab instancu (runtime.unassign) da
# odmah oslobodi GPU slot. T4 je free tier pa ne troši units, ali sesija
# i dalje zauzima jedan slot — auto-shutdown ga oslobađa odmah.
# Postavi na False kad debugiraš ili želiš pregledati output prije gašenja.
AUTO_SHUTDOWN     = True

# ─── Repo ────────────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/domovinatv/fetch.domovina.tv.git"
REPO_PATH = "/content/fetch.domovina.tv"

print("Konfiguracija učitana.")
print(f"  Input:         {INPUT_DIR}")
print(f"  Limit:         {LIMIT if LIMIT else 'sve'}")
print(f"  Batch size:    {BATCH_SIZE}  (paralelni GPU forward pass)")
print(f"  Dry run:       {DRY_RUN}")
print(f"  Auto-shutdown: {AUTO_SHUTDOWN}")
print(f"  Output:        .sortformer.diarized.srt (drop-in kompatibilan s combined notebookom)")


## 1. GPU provjera

**T4 (16 GB) je dovoljan** — Sortformer peak-a na ~2 GB. Ako runtime ima G4/A100, radit će ali bezveze plaćanje units-a.


In [ ]:
import subprocess
out = subprocess.check_output(["nvidia-smi", "-L"]).decode().strip()
print(out)

mem = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total,memory.free", "--format=csv,noheader,nounits"]
).decode().strip()
total, free = [int(x.strip()) for x in mem.split(",")]
print(f"\nVRAM: {free} MB slobodno / {total} MB ukupno ({free/total*100:.0f}%)")

if total < 6000:
    print(f"\nUPOZORENJE: Manje od 6 GB VRAM ({total} MB) — može biti tijesno.")
elif total > 24000:
    print(f"\nℹ️  Imamo {total} MB VRAM-a — više nego dovoljno za Sortformer (~2 GB).")
    print("   Razmisli o downgrade-u na T4 (free) za skaliranje na tisućama fajlova.")


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount(DRIVE_MOUNT_POINT)

import os
assert os.path.isdir(INPUT_DIR), (
    f"Direktorij ne postoji: {INPUT_DIR}\n"
    f"Provjeri DRIVE_DATA_DIR ili da li su WAV+canary.srt uploadani."
)
n_entries = len(os.listdir(INPUT_DIR))
print(f"Drive mountan. INPUT_DIR sadrži: {n_entries} entry-ja.")


## 3. Clone / pull repo


In [ ]:
import os, subprocess

if not os.path.isdir(REPO_PATH):
    print(f"Kloniram repo u {REPO_PATH}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    print("Repo postoji, povlačim najnovije...")
    subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], check=True)

WORKHORSE_SCRIPT = f"{REPO_PATH}/colab_sortformer/diarize_only_sortformer.py"
assert os.path.isfile(WORKHORSE_SCRIPT), f"Nedostaje skripta: {WORKHORSE_SCRIPT}"
print(f"Skripta spremna: {os.path.basename(WORKHORSE_SCRIPT)}")


## 4. Instalacija dependencija (~2-3 min)

NeMo toolkit (sadrži Sortformer). Prvi put gasi kernel radi reloada — nakon restarta klikni **Runtime → Run all** ponovno.


In [ ]:
import os

MARKER = "/content/.sortformer_diarize_only_deps_ok"

def _have_deps():
    try:
        import nemo.collections.asr  # noqa: F401
        from nemo.collections.asr.models import SortformerEncLabelModel  # noqa: F401
        return True
    except Exception as e:
        print(f"deps check: {type(e).__name__}: {e}")
        return False

if os.path.exists(MARKER) and _have_deps():
    print("Dependencies već instalirani i spremni.")
else:
    print("Instaliram NeMo — traje ~2-3 min...")
    get_ipython().system(
        'pip install -qU numpy "nemo_toolkit[asr]" 2>&1 | tail -5'
    )
    open(MARKER, "w").write("ok")
    print("")
    print("=" * 60)
    print(" Instalacija gotova — gasim kernel radi čistog reloada.")
    print(" → Nakon restarta klikni Runtime → Run all ponovno.")
    print("=" * 60)
    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)


In [ ]:
# Bezopasni shimovi za starije NeMo import path-eve
import sys
class _DummyYTTM: pass
sys.modules.setdefault("youtokentome", _DummyYTTM)

import huggingface_hub
if getattr(huggingface_hub, "ModelFilter", None) is None:
    class ModelFilter: pass
    huggingface_hub.ModelFilter = ModelFilter

import torch
print("torch:", torch.__version__,
      "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


## 5. HuggingFace token

Sortformer model card zahtijeva prihvaćanje uvjeta — login s HF tokenom je potreban za `from_pretrained`.


In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN učitan iz Colab Secrets.")
except Exception:
    pass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Upiši HuggingFace token (hf_...): ").strip()

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN nije valjan (mora počinjati s 'hf_')"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("HF_TOKEN spreman.")


## 6. Pregled posla (dry run)

Skenira Drive i prikazuje:
- Koliko WAV-ova ima `.canary.srt` (preduvjet za Pass 2)
- Koliko ima `.sortformer.diarized.srt` (već gotovo)
- Koliko će se obraditi u ovom runu

**WAV-ovi bez `.canary.srt` se ignoriraju** — prvo pokreni Pass 1 (`colab_canary/domovina_tv_fetch.ipynb`) za njih.


In [ ]:
def scan_progress(input_dir):
    wavs, with_canary, with_sortformer = [], [], []
    for root, _, files in os.walk(input_dir, followlinks=True):
        for f in files:
            if f.startswith("._") or not f.endswith(".wav"):
                continue
            p = os.path.join(root, f)
            wavs.append(p)
            if os.path.exists(p + ".canary.srt"):
                with_canary.append(p)
            if os.path.exists(p + ".sortformer.diarized.srt"):
                with_sortformer.append(p)
    return wavs, with_canary, with_sortformer

print("Skeniram Drive (može potrajati ~30s za veliki korpus)...")
wavs, with_canary, with_sortformer = scan_progress(INPUT_DIR)
without_canary = len(wavs) - len(with_canary)
to_process = [w for w in with_canary if w not in set(with_sortformer)]

print(f"\n  Ukupno WAV datoteka:                  {len(wavs)}")
print(f"  Imaju .canary.srt (preduvjet):        {len(with_canary)}")
if without_canary > 0:
    print(f"  ⚠️  Bez .canary.srt (treba Pass 1):  {without_canary}")
print(f"  Imaju .sortformer.diarized.srt:       {len(with_sortformer)}")
print(f"  Za obradu u ovom runu:                {len(to_process)}")

if LIMIT:
    print(f"\n  LIMIT={LIMIT} → obradit će se najviše {LIMIT} fajlova")

# ETA — Sortformer alone vrlo brz (~3-5s/file na T4 za podcast 1-3h)
gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ""
if "t4" in gpu_name:
    sec_per_file = 6   # T4 je sporiji za neuralni inference, ~realtime/N
elif "l4" in gpu_name:
    sec_per_file = 4
elif "a100" in gpu_name or "rtx pro 6000" in gpu_name or "g4" in gpu_name:
    sec_per_file = 3
else:
    sec_per_file = 10
n_to_run = min(LIMIT or 10**9, len(to_process))
if n_to_run:
    eta_min = n_to_run * sec_per_file / 60
    print(f"\n  Procjena trajanja na ovom GPU-u: ~{eta_min:.0f} min")
    print(f"  (heuristika: ~{sec_per_file}s/file × {n_to_run} fajla)")


## 7. Pokreni Sortformer batch (diarize-only, paralelni GPU forward)

Poziva `diarize_only_sortformer.py` koji:

1. Učita Streaming Sortformer 4spk v2.1 (jednom, na početku) — **NE učitava Canary**
2. Sortira fajlove po veličini (minimizira pad-to-longest waste u batchu)
3. Za svaki batch od `BATCH_SIZE` fajlova: jedan paralelni GPU forward pass
4. Per-file: parsira `.canary.srt` → merge speakera (best-overlap) → `.sortformer.diarized.srt`
5. Heartbeat svakih 60s, ETA na osnovu prosjeka, fallback na per-file ako batch faila s OOM

**Tuning**: ako vidiš `CUDA OOM` ili System RAM blizu 12 GB → smanji `BATCH_SIZE` u ćeliji 0. Ako GPU ostaje idle (`nvidia-smi` < 50% util tijekom inferencije) → povećaj.


In [ ]:
import shlex

cmd = [
    "python", "-u", WORKHORSE_SCRIPT,
    "--input-dir", INPUT_DIR,
    "--batch-size", str(BATCH_SIZE),
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if DRY_RUN:
    cmd += ["--dry-run"]

print(">", " ".join(shlex.quote(c) for c in cmd))
print()
get_ipython().system(" ".join(shlex.quote(c) for c in cmd))


## 8. Sažetak — što je novo na Drive-u

Delta novih `.sortformer.diarized.srt` u ovom runu.


In [ ]:
wavs2, with_canary2, with_sortformer2 = scan_progress(INPUT_DIR)
new_sortformer = len(with_sortformer2) - len(with_sortformer)

print("─" * 60)
print(f"  Novih .sortformer.diarized.srt:  +{new_sortformer}")
print("─" * 60)
print(f"  Stanje na Drive-u:")
print(f"    {len(wavs2)} WAV ukupno")
print(f"    {len(with_canary2)}/{len(wavs2)} ima .canary.srt")
print(f"    {len(with_sortformer2)}/{len(with_canary2)} sortformer diarized")
print()
remaining = len(with_canary2) - len(with_sortformer2)
if remaining > 0:
    print(f"  Još {remaining} fajlova s .canary.srt čeka diarizaciju —")
    print(f"  pokreni notebook ponovno (idempotentno).")
else:
    print("  ✅ Sve .canary.srt datoteke su diarized!")


## 9. Auto-shutdown — pusti Colab instancu

Ako je `AUTO_SHUTDOWN = True` (cell 0), poziva `runtime.unassign()` koji **odmah** terminira VM. T4 je free tier (0 compute units), ali sesija i dalje zauzima jedan GPU slot dok ne odspoji — ovo ga oslobađa odmah umjesto da čekaš idle timeout (~90 min).

Postavi `AUTO_SHUTDOWN = False` na vrhu kad debugiraš ili kad želiš ručno pregledati output prije gašenja.


In [ ]:
if AUTO_SHUTDOWN:
    print("AUTO_SHUTDOWN=True → gasim Colab runtime za 5s…")
    print("(Ako želiš odustati: Runtime → Interrupt execution sada.)")
    import time
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
else:
    print("AUTO_SHUTDOWN=False → instanca ostaje živa.")
    print("Ručno gašenje: Runtime → Disconnect and delete runtime")
    print("(ili pokreni: from google.colab import runtime; runtime.unassign())")
